In [16]:
import numpy as np
import os
import pandas as pd
from scipy.optimize import curve_fit
from LPPL_codex import LPPLModel
import plotly.graph_objects as go


In [17]:
in_dir = './result/weekly'
ticker = '/GC'

in_path = os.path.join(in_dir, '_GC_weekly_close.parquet')
df_result = pd.read_parquet(in_path)

In [18]:
# Inspect the data
print("Data shape:", df_result.shape)
print("Columns:", df_result.columns.tolist())
print("Data types:")
print(df_result.dtypes)
print("First few rows:")
print(df_result.head())

# Check for missing values
print("\nMissing values per column:")
print(df_result.isnull().sum())

# Handle missing values - drop rows with missing tc or key columns
key_columns = ['window_start', 'window_end', 'tc']
df_result = df_result.dropna(subset=key_columns)

print("Data shape after cleaning:", df_result.shape)

Data shape: (105237, 20)
Columns: ['window_end', 'window_start', 'window_size', 'success', 'tc', 'tc_index_ahead', 'tc_date', 'm', 'omega', 'A', 'B', 'C1', 'C2', 'C', 'phi', 'rmse', 'sse', 'r2', 'n_obs', 'message']
Data types:
window_end        datetime64[ns, UTC]
window_start      datetime64[ns, UTC]
window_size                     int64
success                          bool
tc                            float64
tc_index_ahead                float64
tc_date           datetime64[ns, UTC]
m                             float64
omega                         float64
A                             float64
B                             float64
C1                            float64
C2                            float64
C                             float64
phi                           float64
rmse                          float64
sse                           float64
r2                            float64
n_obs                           int64
message                        object
dtype: object

In [19]:

# Filter for r2 > 0.95
pivot_table = df_result[df_result['r2'] > 0.95].pivot_table(index='window_end', columns='window_size', values='tc', aggfunc='mean')

# Use numeric indices for the surface axes
x_vals = list(range(len(pivot_table.columns)))
y_vals = list(range(len(pivot_table.index)))

# window_size is numeric — use raw values as labels
x_labels = [str(v) for v in pivot_table.columns]
y_labels = [pd.Timestamp(d).strftime('%Y-%m') for d in pivot_table.index]

# Subsample tick labels to avoid crowding
x_step = max(1, len(x_vals) // 10)
y_step = max(1, len(y_vals) // 10)
x_tickvals = x_vals[::x_step]
x_ticktext = x_labels[::x_step]
y_tickvals = y_vals[::y_step]
y_ticktext = y_labels[::y_step]

X, Y = np.meshgrid(x_vals, y_vals)
Z = pivot_table.values

# Create the interactive 3D plot
fig = go.Figure(data=[go.Surface(x=X, y=Y, z=Z, colorscale='Viridis')])

fig.update_layout(
    title='Interactive Surface Plot of tc by Window End and Window Size',
    scene=dict(
        xaxis=dict(title='Window Size', tickvals=x_tickvals, ticktext=x_ticktext),
        yaxis=dict(title='Window End', tickvals=y_tickvals, ticktext=y_ticktext),
        zaxis_title='tc'
    ),
    width=800,
    height=600
)

fig.show()

In [20]:

# Filter for r2 > 0.95
pivot_table = df_result[df_result['r2'] > 0.95].pivot_table(index='window_start', columns='window_size', values='tc', aggfunc='mean')

# Use numeric indices for the surface axes
x_vals = list(range(len(pivot_table.columns)))
y_vals = list(range(len(pivot_table.index)))

# window_size is numeric — use raw values as labels
x_labels = [str(v) for v in pivot_table.columns]
y_labels = [pd.Timestamp(d).strftime('%Y-%m') for d in pivot_table.index]

# Subsample tick labels to avoid crowding
x_step = max(1, len(x_vals) // 10)
y_step = max(1, len(y_vals) // 10)
x_tickvals = x_vals[::x_step]
x_ticktext = x_labels[::x_step]
y_tickvals = y_vals[::y_step]
y_ticktext = y_labels[::y_step]

X, Y = np.meshgrid(x_vals, y_vals)
Z = pivot_table.values

# Create the interactive 3D plot
fig = go.Figure(data=[go.Surface(x=X, y=Y, z=Z, colorscale='Viridis')])

fig.update_layout(
    title='Interactive Surface Plot of tc by Window End and Window Size',
    scene=dict(
        xaxis=dict(title='Window Size', tickvals=x_tickvals, ticktext=x_ticktext),
        yaxis=dict(title='Window End', tickvals=y_tickvals, ticktext=y_ticktext),
        zaxis_title='tc'
    ),
    width=800,
    height=600
)

fig.show()

In [21]:

# Pivot the data to create a 2D grid (aggfunc='mean' handles any duplicate combinations)
pivot_table = df_result[df_result['r2'] > 0.97].pivot_table(index='window_end', columns='window_size', values='omega', aggfunc='mean')

# Use numeric indices for the surface axes
x_vals = list(range(len(pivot_table.columns)))
y_vals = list(range(len(pivot_table.index)))

# window_size is numeric — use raw values as labels
x_labels = [str(v) for v in pivot_table.columns]
y_labels = [pd.Timestamp(d).strftime('%Y-%m') for d in pivot_table.index]

# Subsample tick labels to avoid crowding
x_step = max(1, len(x_vals) // 10)
y_step = max(1, len(y_vals) // 10)
x_tickvals = x_vals[::x_step]
x_ticktext = x_labels[::x_step]
y_tickvals = y_vals[::y_step]
y_ticktext = y_labels[::y_step]

X, Y = np.meshgrid(x_vals, y_vals)
Z = pivot_table.values

# Create the interactive 3D plot
fig = go.Figure(data=[go.Surface(x=X, y=Y, z=Z, colorscale='Viridis')])

fig.update_layout(
    title='Interactive Surface Plot of omega by Window End and Window Size',
    scene=dict(
        xaxis=dict(title='Window Size', tickvals=x_tickvals, ticktext=x_ticktext),
        yaxis=dict(title='Window End', tickvals=y_tickvals, ticktext=y_ticktext),
        zaxis_title='omega'
    ),
    width=800,
    height=600
)

fig.show()

In [ ]:
from ipywidgets import interact, FloatSlider
import plotly.graph_objects as go

# Global figure to maintain state
global_fig = None

def plot_tc_by_r2_threshold(r2_threshold):
    global global_fig
    
    # Filter data by r2 threshold
    filtered_df = df_result[df_result['r2'] > r2_threshold]
    
    if filtered_df.empty:
        print(f"No data available for r2 > {r2_threshold}")
        return
    
    # Pivot the data to create a 2D grid
    pivot_table = filtered_df.pivot_table(index='window_end', columns='window_size', values='tc', aggfunc='mean')
    
    # Use numeric indices for the surface axes
    x_vals = list(range(len(pivot_table.columns)))
    y_vals = list(range(len(pivot_table.index)))
    
    # window_size is numeric — use raw values as labels
    x_labels = [str(v) for v in pivot_table.columns]
    y_labels = [pd.Timestamp(d).strftime('%Y-%m') for d in pivot_table.index]
    
    # Subsample tick labels to avoid crowding
    x_step = max(1, len(x_vals) // 10)
    y_step = max(1, len(y_vals) // 10)
    x_tickvals = x_vals[::x_step]
    x_ticktext = x_labels[::x_step]
    y_tickvals = y_vals[::y_step]
    y_ticktext = y_labels[::y_step]
    
    X, Y = np.meshgrid(x_vals, y_vals)
    Z = pivot_table.values
    
    # Update existing figure or create new one
    if global_fig is None:
        # Create the interactive 3D plot
        global_fig = go.Figure(data=[go.Surface(x=X, y=Y, z=Z, colorscale='Viridis')])
    else:
        # Update the surface data
        global_fig.data[0].x = X
        global_fig.data[0].y = Y
        global_fig.data[0].z = Z
    
    global_fig.update_layout(
        title=f'Interactive Surface Plot of tc by Window End and Window Size (r2 > {r2_threshold})',
        scene=dict(
            xaxis=dict(title='Window Size', tickvals=x_tickvals, ticktext=x_ticktext),
            yaxis=dict(title='Window End', tickvals=y_tickvals, ticktext=y_ticktext),
            zaxis_title='tc',
            uirevision='constant'  # Preserve camera position across updates
        ),
        width=800,
        height=600
    )
    
    global_fig.show()

# Create interactive slider for r2 threshold
r2_slider = FloatSlider(
    value=0.95,
    min=0.0,
    max=1.0,
    step=0.01,
    description='R² Threshold:',
    continuous_update=False
)

interact(plot_tc_by_r2_threshold, r2_threshold=r2_slider)

Output()

FloatSlider(value=0.95, continuous_update=False, description='R² Threshold:', max=1.0, step=0.01)